## 09 · ML-assisted literature coding (WP1 / D1.2)

Thin driver over `gif.lit_ml`: trains TF-IDF text classifiers on the **manually coded corpus** (ground truth from [10.5281/zenodo.20744025](https://doi.org/10.5281/zenodo.20744025)) so future literature batches can be **pre-coded automatically** and only reviewed by hand.

Tasks (derived to respect actual label support):
- `sector` — AGRI vs MINING (366 papers)
- `region` — EU vs non-EU (366 papers)
- `relevance` — high vs medium/low (**only ~101 labeled papers**)
- `code:<dimension>:<code>` — one binary task per manual code with ≥15 papers

Each task is evaluated with stratified cross-validation (macro-F1) across four models; the best per task is refit on all labeled data and stored in `notebooks/models/literature_coder.pkl`.

<div class="alert alert-block alert-warning"><b>Caveats:</b> (1) 366 papers is a small corpus — treat predictions as triage suggestions, not replacements for manual coding. (2) The <code>region</code> task scores ≈1.0 because non-EU papers almost always name Kenya/Canada in their abstracts — the task is legitimately easy, not leaky. (3) <code>relevance</code> is weak (~0.57 macro-F1) due to few labels.</div>

In [ ]:
import sys
from pathlib import Path

# Make the repo importable: `helper` at the root, `gif` under `src/`.
repo_root = Path.cwd().parents[0]
for p in (str(repo_root), str(repo_root / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from gif.lit_ml import run_literature_coding

In [ ]:
result = run_literature_coding(
    repo_root / "data/processed/literature",
    repo_root / "data/results/literature",
    repo_root / "notebooks/models",
)
result.best[["task", "model", "f1_macro", "accuracy", "n"]]

### Evaluation figures

In [ ]:
from IPython.display import Image, display
for name in ["lit_coding_f1_by_task.png", "lit_coding_sector_confusion.png"]:
    f = repo_root / "data/results/literature" / name
    if f.exists():
        display(Image(filename=str(f)))

### Pre-code new papers

Apply the saved bundle to any frame with `title` (+ `abstract`) columns — e.g. the next D1.x literature batch:

In [ ]:
import pandas as pd
from gif.lit_ml import predict_codes

demo = pd.DataFrame({
    "title": ["Lignite mining rehabilitation in post-transition regions"],
    "abstract": ["Open-pit coal extraction sites in Europe require "
                 "long-term land reclamation strategies..."],
})
predict_codes(repo_root / "notebooks/models/literature_coder.pkl", demo).T